[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jeremylongshore/claude-code-plugins-plus-skills/blob/main/tutorials/skills/02-skill-anatomy.ipynb)

# Skill Anatomy: Deep Dive into SKILL.md Structure

**Learning Path**: Skills → Plugins → Orchestration  
**Level**: Beginner-Intermediate  
**Time**: 30 minutes  
**Prerequisites**: [01-what-is-skill.ipynb](01-what-is-skill.ipynb)

---

## What You'll Learn

- Complete SKILL.md frontmatter structure
- Required vs optional fields
- Body structure: the 7 required sections
- Tool authorization patterns (with Bash scoping)
- Model selection strategies
- Build and validate real skills

---

## SKILL.md: Two Parts

Every SKILL.md has exactly **two sections**:

```markdown
---
[FRONTMATTER: YAML configuration]
---

[BODY: Markdown instructions]
```

### Part 1: Frontmatter (YAML)

**What it tells Claude**:
- **Name**: What to call this skill
- **Description**: When to use it (triggers + use cases)
- **Tools**: Which tools are pre-authorized
- **Model**: Which Claude model to use (optional override)
- **Metadata**: Version, author, license

### Part 2: Body (Markdown)

**What it tells Claude**:
- **How** to execute the skill
- 7 required sections: Overview, Prerequisites, Instructions, Output, Error Handling, Examples, Resources
- Purpose statement (1-2 sentences, ≤400 chars)
- Step-by-step instructions
- Examples and templates

In [ ]:
# Visualize the structure
structure = {
    "FRONTMATTER (YAML)": {
        "tells_claude": "WHAT and WHEN",
        "fields": [
            "name (required)",
            "description (required)",
            "allowed-tools (required, CSV)",
            "version (required)",
            "author (required)",
            "license (required)",
            "--- optional below ---",
            "model (optional)",
            "context (optional)",
            "agent (optional)",
            "user-invocable (optional)",
            "argument-hint (optional)",
            "hooks (optional)",
            "compatibility (optional)",
            "compatible-with (optional)",
            "tags (optional)",
        ]
    },
    "BODY (Markdown)": {
        "tells_claude": "HOW to execute",
        "contains": [
            "Overview (required section)",
            "Prerequisites (required section)",
            "Instructions (required section)",
            "Output (required section)",
            "Error Handling (required section)",
            "Examples (required section)",
            "Resources (required section)",
        ]
    }
}

print("SKILL.MD ANATOMY")
print("=" * 60)
for section, details in structure.items():
    print(f"\n{section}")
    print(f"  Purpose: {details['tells_claude']}")
    
    key = 'fields' if 'fields' in details else 'contains'
    print(f"  {key.title()}:")
    for item in details[key]:
        print(f"    - {item}")

print("\n" + "=" * 60)
print("Frontmatter = Configuration, Body = Instructions")

---

## Frontmatter: Required Fields (6 Required)

The Intent Solutions standard requires exactly **6 fields** in every SKILL.md frontmatter:

### 1. `name` (String)

**Format**: `kebab-case` (lowercase, hyphens only)
**Pattern**: `^[a-z0-9-]+$`
**Max length**: 64 characters
**Reserved**: Cannot contain "claude" or "anthropic"

```yaml
name: bigquery-migrations      # CORRECT
name: BigQueryMigrations       # WRONG (camelCase)
name: bigquery_migrations      # WRONG (snake_case)
name: claude-helper            # WRONG (reserved word)
```

### 2. `description` (String, Multiline)

**Must include**:
1. What the skill does (1 sentence)
2. "Use when..." phrase (triggers automatic invocation)
3. 2-6 trigger phrases (helps Claude match intent)

```yaml
description: |
  Generate BigQuery schema migrations with validation.
  Use when: migrating schemas, adding fields, changing types.
  Triggers: migration, schema change, add field, alter table.
```

### 3. `allowed-tools` (String, CSV)

**CRITICAL**: Must be CSV string, NOT YAML array

```yaml
allowed-tools: "Read,Write,Grep,Bash(git:*)"  # CORRECT (CSV)
allowed-tools: [Read, Write, Grep]             # WRONG (array)
```

**Bash scoping is MANDATORY** -- never use bare `Bash`:
```yaml
allowed-tools: "Bash(git:*),Bash(npm:*)"  # CORRECT (scoped)
allowed-tools: "Bash"                     # WRONG (unscoped)
```

### 4. `version` (String, SemVer)

**Format**: `MAJOR.MINOR.PATCH` (3 parts)

```yaml
version: 1.0.0      # CORRECT
version: 1.0        # WRONG (only 2 parts)
version: "1.0.0"    # CORRECT (quoted is fine)
```

### 5. `author` (String)

**Format**: `Name <email>`

```yaml
author: Jane Developer <jane@example.com>  # CORRECT
author: Jane Developer                      # WRONG (no email)
```

### 6. `license` (String, SPDX)

```yaml
license: MIT
license: Apache-2.0
license: GPL-3.0
```

> **Note**: `tags` is OPTIONAL -- it helps with categorization and discovery but is not required.

In [ ]:
# Build a valid frontmatter
def build_frontmatter(name, description, tools, version="1.0.0", 
                     license="MIT", author="Developer <dev@example.com>",
                     tags=None):
    """Build a standards-compliant SKILL.md frontmatter.
    
    6 required fields: name, description, allowed-tools, version, author, license.
    tags is optional.
    """
    
    frontmatter = f"""---
name: {name}
description: |
  {description}
allowed-tools: "{tools}"
version: {version}
author: {author}
license: {license}"""
    
    if tags:
        frontmatter += f"\ntags: {tags}"
    
    frontmatter += "\n---"
    
    return frontmatter

# Example: Build a skill frontmatter
example = build_frontmatter(
    name="code-reviewer",
    description="""Review code for quality, security, and best practices.
  Use when: reviewing pull requests, auditing code, checking standards.
  Triggers: review code, code quality, security audit, check pr.""",
    tools="Read,Grep,Bash(git:*)",
    tags=["code-review", "quality", "security"]
)

print("GENERATED FRONTMATTER:")
print("=" * 60)
print(example)
print("=" * 60)
print("\nThis follows all 6 required fields per the skill standard!")

---

## Frontmatter: Optional Fields

### `model` (String)

**Override default model for this skill**

```yaml
model: sonnet    # Claude Sonnet (default, complex reasoning)
model: haiku     # Claude Haiku (faster, cheaper, simple tasks)
model: opus      # Claude Opus 4.6 (highest capability)
```

**When to use**:
- `haiku`: Simple, deterministic tasks (fast)
- `sonnet`: Complex reasoning, analysis (default)
- `opus`: Maximum capability, difficult multi-step reasoning

### `context` and `agent`

**Run the skill in a subagent (forked context)**

```yaml
context: fork        # Run in a subagent
agent: Explore       # Subagent type
```

### `user-invocable` and `argument-hint`

```yaml
user-invocable: false          # Hide from / menu (auto-trigger only)
argument-hint: "<file-path>"   # Autocomplete hint for slash menu
```

### `hooks`

```yaml
hooks:
  pre-tool-call: ...    # Lifecycle hooks
```

### `compatibility` and `compatible-with`

```yaml
compatibility: "Node.js >= 18"                        # Environment requirements
compatible-with: claude-code, cursor, windsurf, cline  # Platform compatibility
```

Valid `compatible-with` values: `claude-code`, `cursor`, `windsurf`, `cline`

### `tags` (Array, Optional)

**Purpose**: Categorization and discovery (NOT required)

```yaml
tags: ["database", "bigquery", "migration", "schema"]
```

In [ ]:
# Tool authorization patterns
tool_patterns = {
    "Read-only analysis": "Read,Grep,Glob",
    
    "File manipulation": "Read,Write,Edit,Grep,Glob",
    
    "Git operations": "Read,Write,Bash(git:*)",
    
    "Package management": "Read,Bash(npm:*),Bash(pip:*)",
    
    "Database operations": "Read,Write,Bash(psql:*),Bash(mysql:*)",
    
    "Web research": "Read,WebFetch,WebSearch",
    
    "Full access (use sparingly)": "Read,Write,Edit,Grep,Glob,Bash(git:*),Bash(npm:*)",
}

print("COMMON TOOL AUTHORIZATION PATTERNS")
print("=" * 60)
for use_case, tools in tool_patterns.items():
    print(f"\n{use_case}:")
    print(f"  allowed-tools: \"{tools}\"")

print("\n" + "=" * 60)
print("Principle: Grant minimum tools needed")
print("  Read-only skills don't need Write/Edit")
print("  ALWAYS scope Bash: Bash(git:*), never bare Bash")

---

## Body Structure: The 7 Required Sections

### Size Limits

**SKILL.md body target**:
- **Target**: ≤150 lines
- **Maximum**: 500 lines absolute max

**Progressive Disclosure Architecture (PDA)**: If your skill content exceeds 150 lines, move heavy implementation details into `references/implementation.md` and reference it via `${CLAUDE_SKILL_DIR}/references/implementation.md`. The SKILL.md stays lean; the agent reads references on demand.

### Purpose Statement Rules

The first paragraph after `# Title`, or content in `## Overview`, serves as the **purpose statement**:
- Must be 1-2 sentences
- Must be ≤400 characters
- Explains what the skill does and when to use it

### The 7 Required Sections

```markdown
# [Skill Name]

## Overview

Brief purpose statement (1-2 sentences, ≤400 chars).

## Prerequisites

What must exist before running (files, env vars, tools, etc.)

## Instructions

### Step 1: [Action]

Detailed instructions...

### Step 2: [Action]

Detailed instructions...

## Output

What the skill should produce (JSON, files, reports, etc.)

## Error Handling

How to handle common errors and edge cases.

## Examples

Working examples of skill execution.

## Resources

- Related documentation
- See `${CLAUDE_SKILL_DIR}/references/implementation.md` for details
```

### Code Block Rules

Bash code blocks with dangerous commands (`rm`, `curl`, `pip`, etc.) must include `set -euo pipefail`:

````markdown
```bash
set -euo pipefail
curl -sSL https://example.com/install.sh | bash
```
````

Magic numbers (>=200) must have inline comments containing the number:

```python
timeout = 300  # 300 second timeout for large repos
```

In [ ]:
# Build a complete SKILL.md with all 7 required sections
def build_complete_skill(name, description, tools, body_content,
                         version="1.0.0", license="MIT",
                         author="Developer <dev@example.com>",
                         tags=None, model=None):
    """Build a complete standards-compliant SKILL.md file.
    
    Frontmatter: 6 required fields (name, description, allowed-tools,
                 version, author, license) plus optional fields.
    Body: 7 required sections (Overview, Prerequisites, Instructions,
          Output, Error Handling, Examples, Resources).
    """
    
    # Build frontmatter
    frontmatter = f"""---
name: {name}
description: |
  {description}
allowed-tools: "{tools}"
version: {version}
author: {author}
license: {license}"""
    
    if model:
        frontmatter += f"\nmodel: {model}"
    
    if tags:
        frontmatter += f"\ntags: {tags}"
    
    frontmatter += "\n---"
    
    # Combine frontmatter + body
    complete_skill = f"{frontmatter}\n\n{body_content}"
    
    return complete_skill

# Example: Build a complete skill with all 7 required body sections
body = """# Git Commit Message Generator

## Overview

Generates conventional commit messages following team standards. Use when creating commits or writing commit messages.

## Prerequisites

- Git repository initialized
- Staged changes exist (`git diff --staged` is non-empty)

## Instructions

### Step 1: Analyze Changes
Use `git diff --staged` to see what changed.

### Step 2: Determine Type
- feat: New feature
- fix: Bug fix
- docs: Documentation
- refactor: Code refactoring

### Step 3: Write Message
Format: `type(scope): description`

## Output

Return the commit message as plain text, formatted as:
```
type(scope): short description

Optional longer body explaining the change.
```

## Error Handling

- If no staged changes: prompt user to stage files first
- If scope is ambiguous: ask user to clarify

## Examples

Input: staged change adds a new login endpoint
Output: `feat(auth): add login endpoint with JWT validation`

## Resources

- [Conventional Commits](https://conventionalcommits.org)
- See `${CLAUDE_SKILL_DIR}/references/implementation.md` for advanced patterns
"""

skill = build_complete_skill(
    name="git-commit-helper",
    description="""Generate conventional commit messages.
  Use when: creating commits, writing commit messages.
  Triggers: commit message, generate commit, conventional commits.""",
    tools="Read,Bash(git:*)",
    body_content=body,
    tags=["git", "commits", "productivity"],
    model="haiku"  # Fast for simple task
)

print("COMPLETE SKILL.MD:")
print("=" * 60)
print(skill)
print("=" * 60)

# Count lines to verify size
lines = body.strip().split('\n')
print(f"\nBody line count: {len(lines)} (target: <=150, max: 500)")
print("Ready to save as SKILL.md!")

---

## Validate Your Skill

In [ ]:
import re

def validate_skill(skill_content):
    """Validate skill against the Intent Solutions standard."""
    errors = []
    warnings = []
    
    # Split frontmatter and body
    parts = skill_content.split('---')
    if len(parts) < 3:
        errors.append("CRITICAL: Invalid format - need frontmatter between ---")
        return errors, warnings
    
    frontmatter_raw = parts[1].strip()
    body = '---'.join(parts[2:]).strip()
    
    # Parse frontmatter (simple YAML)
    frontmatter = {}
    for line in frontmatter_raw.split('\n'):
        if ':' in line and not line.startswith(' '):
            key, value = line.split(':', 1)
            frontmatter[key.strip()] = value.strip()
    
    # 6 required fields (tags is NOT required)
    required = ["name", "description", "allowed-tools", "version", "author", "license"]
    for field in required:
        if field not in frontmatter:
            errors.append(f"CRITICAL: Missing required field '{field}'")
    
    # Name validation
    if "name" in frontmatter:
        name = frontmatter["name"].strip('"')
        if not re.match(r'^[a-z0-9-]+$', name):
            errors.append(f"CRITICAL: Name must be kebab-case: {name}")
        if "claude" in name or "anthropic" in name:
            errors.append(f"CRITICAL: Name cannot contain 'claude' or 'anthropic'")
    
    # allowed-tools validation
    if "allowed-tools" in frontmatter:
        tools = frontmatter["allowed-tools"]
        if not tools.startswith('"'):
            errors.append("CRITICAL: allowed-tools must be quoted CSV string")
        # Check for bare Bash (not scoped)
        tools_unquoted = tools.strip('"')
        tool_list = [t.strip() for t in tools_unquoted.split(',')]
        if "Bash" in tool_list:
            errors.append("CRITICAL: Bash must be scoped (e.g., Bash(git:*)), never bare Bash")
    
    # Version validation
    if "version" in frontmatter:
        version = frontmatter["version"].strip('"')
        if not re.match(r'^\d+\.\d+\.\d+$', version):
            errors.append(f"CRITICAL: Version must be SemVer (x.y.z): {version}")
    
    # Model validation - sonnet, haiku, opus are all valid
    if "model" in frontmatter:
        model = frontmatter["model"].strip('"')
        valid_models = {"sonnet", "haiku", "opus"}
        if model not in valid_models:
            errors.append(f"CRITICAL: Model must be one of {valid_models}: {model}")
    
    # compatible-with validation
    if "compatible-with" in frontmatter:
        platforms = frontmatter["compatible-with"].strip('"')
        valid_platforms = {"claude-code", "cursor", "windsurf", "cline"}
        for p in [x.strip() for x in platforms.split(',')]:
            if p and p not in valid_platforms:
                errors.append(f"CRITICAL: Invalid compatible-with value '{p}'. Valid: {valid_platforms}")
    
    # Body size check (PDA: target 150 lines, max 500)
    lines = body.split('\n')
    words = body.split()
    if len(lines) > 500:
        errors.append(f"CRITICAL: Body has {len(lines)} lines (max 500)")
    elif len(lines) > 150:
        warnings.append(f"Body has {len(lines)} lines (target: <=150). Consider PDA: move content to references/implementation.md")
    
    # Check for required body sections
    required_sections = ["Overview", "Prerequisites", "Instructions", "Output", "Error Handling", "Examples", "Resources"]
    for section in required_sections:
        # Look for ## Section or ## section (case-insensitive heading)
        pattern = rf'^##\s+{re.escape(section)}'
        if not re.search(pattern, body, re.MULTILINE | re.IGNORECASE):
            warnings.append(f"Missing recommended section: ## {section}")
    
    # Check for purpose statement length (first paragraph after # Title)
    title_match = re.search(r'^#\s+.+\n\n(.+?)(?:\n\n|\n##)', body, re.MULTILINE | re.DOTALL)
    if not title_match:
        # Try ## Overview section
        overview_match = re.search(r'##\s+Overview\s*\n\n(.+?)(?:\n\n|\n##)', body, re.MULTILINE | re.DOTALL)
        if overview_match:
            purpose = overview_match.group(1).strip()
            if len(purpose) > 400:
                warnings.append(f"Purpose statement is {len(purpose)} chars (max 400)")
    
    # Check for ${CLAUDE_PLUGIN_ROOT} (should be ${CLAUDE_SKILL_DIR})
    if '${CLAUDE_PLUGIN_ROOT}' in skill_content:
        errors.append("CRITICAL: Use ${CLAUDE_SKILL_DIR} instead of ${CLAUDE_PLUGIN_ROOT}")
    
    # Check for dangerous bash blocks without set -euo pipefail
    dangerous_cmds = ['rm ', 'curl ', 'pip ', 'wget ']
    bash_blocks = re.findall(r'```bash\n(.*?)```', body, re.DOTALL)
    for block in bash_blocks:
        has_dangerous = any(cmd in block for cmd in dangerous_cmds)
        if has_dangerous and 'set -euo pipefail' not in block:
            warnings.append("Bash block with dangerous commands should include 'set -euo pipefail'")
    
    return errors, warnings

# Test with our example
errors, warnings = validate_skill(skill)

print("VALIDATION RESULTS (Intent Solutions standard)")
print("=" * 60)
if not errors and not warnings:
    print("PASSED - Skill follows the skill standard!")
else:
    if errors:
        print("\nCRITICAL ERRORS:")
        for error in errors:
            print(f"   {error}")
    if warnings:
        print("\nWARNINGS:")
        for warning in warnings:
            print(f"   {warning}")

print("\n" + "=" * 60)

---

## Key Takeaways

### Frontmatter (YAML)

1. **6 required fields**: name, description, allowed-tools, version, author, license
2. **Name**: kebab-case, no reserved words
3. **allowed-tools**: CSV string (not array), Bash must be scoped (`Bash(git:*)`, never bare `Bash`)
4. **Version**: SemVer (x.y.z)
5. **Description**: Include "Use when..." and trigger phrases
6. **tags**: Optional, for categorization and discovery

### Body (Markdown)

1. **Size**: Target ≤150 lines (500 max). Use PDA for heavy content: `${CLAUDE_SKILL_DIR}/references/implementation.md`
2. **7 required sections**: Overview, Prerequisites, Instructions, Output, Error Handling, Examples, Resources
3. **Purpose statement**: 1-2 sentences, ≤400 chars (in Overview or first paragraph)
4. **Code blocks**: Dangerous bash commands need `set -euo pipefail`. Magic numbers (>=200) need inline comments.

### Best Practices

- **Minimum tools**: Grant only what's needed
- **Model selection**: `haiku` for simple, `sonnet` for complex, `opus` for maximum capability
- **Keep it focused**: One skill = one clear purpose
- **Path variable**: Always use `${CLAUDE_SKILL_DIR}` (not `${CLAUDE_PLUGIN_ROOT}`)
- **Test thoroughly**: Validate before deployment

---

## Next Steps

**You now understand skill anatomy!**

1. **[03-build-your-first-skill.ipynb](03-build-your-first-skill.ipynb)** - Create a real skill from scratch
2. **[04-advanced-patterns.ipynb](04-advanced-patterns.ipynb)** - Multi-phase workflows
3. **[05-skill-standards.ipynb](05-skill-standards.ipynb)** - Full validation system

---

*Intent Solutions Standard Compliant -- Version 1.0.0 -- MIT License*